# 15 MCU / TinyML 极低功耗部署

## 定位

手机 SoC（GB 级内存）之外，还有 Cortex-M + NPU（如 Arm Ethos-U）上的 **KB–MB 级** 模型：关键词检测、传感器异常检测、简单分类。ExecuTorch / TFLite Micro / CMSIS-NN 是主流路径。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(42)
np.random.seed(42)
print(f"PyTorch {torch.__version__} | CUDA={torch.cuda.is_available()}")

## 15.1 资源预算对比

In [ ]:
TIERS = [
    ("旗舰手机 NPU", "4-16GB", "10-60 TOPS", "1B-7B LLM"),
    ("MCU + Ethos-U", "256KB-2MB SRAM", "0.05-0.5 TOPS", "10K-1M 参数"),
    ("纯 MCU", "64-512KB", "无 NPU", "1K-100K 参数"),
]
for name, mem, tops, model in TIERS:
    print(f"{name:14s} mem={mem:14s} compute={tops:14s} 模型={model}")

## 15.2 微型网络：深度可分离卷积 KWS 示意

In [ ]:
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, cin, cout, k=3, stride=1):
        super().__init__()
        self.dw = nn.Conv1d(cin, cin, k, stride=stride, padding=k//2, groups=cin, bias=False)
        self.pw = nn.Conv1d(cin, cout, 1, bias=False)
        self.bn = nn.BatchNorm1d(cout)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.act(self.bn(self.pw(self.dw(x))))


class TinyKWS(nn.Module):
    """关键词检测玩具模型：输入 [B, 1, T] 波形特征。"""
    def __init__(self, n_classes=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, 8, 3, padding=1), nn.ReLU(),
            DepthwiseSeparableConv(8, 16, stride=2),
            DepthwiseSeparableConv(16, 32, stride=2),
            nn.AdaptiveAvgPool1d(1),
        )
        self.fc = nn.Linear(32, n_classes)

    def forward(self, x):
        h = self.net(x).squeeze(-1)
        return self.fc(h)


model = TinyKWS()
x = torch.randn(4, 1, 16000 // 10)  # 100ms @16kHz 示意
y = model(x)
n_params = sum(p.numel() for p in model.parameters())
print(f"输出: {tuple(y.shape)}  参数量: {n_params:,}  约 {n_params*1/1024:.1f} KB (INT8)")

## 15.3 INT8 权重量化与静态内存规划

In [ ]:
def quantize_model_int8(module: nn.Module):
    qstate = {}
    for name, p in module.state_dict().items():
        if p.dtype.is_floating_point:
            scale = p.abs().max().clamp(min=1e-8) / 127
            q = torch.round(p / scale).clamp(-127, 127).to(torch.int8)
            qstate[name] = (q, scale)
        else:
            qstate[name] = p
    return qstate


def footprint_bytes(qstate: dict) -> int:
    total = 0
    for v in qstate.values():
        if isinstance(v, tuple):
            total += v[0].numel() * 1 + 4  # int8 + scale fp32
        else:
            total += v.numel() * v.element_size()
    return total


qs = quantize_model_int8(model)
print(f"INT8 权重占用约: {footprint_bytes(qs)/1024:.2f} KB")
print("MCU 部署要点: AOT 内存规划、无动态分配、算子裁剪、CMSIS-NN/Ethos-U delegate")

## 15.4 与手机 LLM 部署的差异

| 维度 | 手机 LLM | MCU TinyML |
|------|---------|------------|
| 模型 | Transformer SLM | DS-CNN / 小 MLP |
| 运行时 | llama.cpp / ExecuTorch | ExecuTorch / TFLM |
| 内存 | GB | KB–MB |
| 延迟 | 几十~几百 ms/token | 几~几十 ms/帧 |
| 典型任务 | 对话 / Agent | KWS / 传感 |

ExecuTorch 可用同一套 `torch.export` 流程下沉到 Ethos-U，是连接两档设备的桥梁。